# Flow Shop Scheduling (FSSP) — Julia

> Material de aula — o **caso especial** do job shop em que **todos os jobs visitam as máquinas
> na mesma ordem** ($M_1 \to M_2 \to \dots \to M_m$). Usamos o dataset de Elkamel & Gharbi
> (`akilelkamel/fssp-dataset`) e resolvemos a minimização do **makespan** com JuMP + HiGHS.

## O problema

No **Flow Shop Scheduling Problem (FSSP)** há $n$ jobs e $m$ máquinas em série. Cada job $j$ tem
um tempo de processamento $p_{j,i}$ na máquina $i$ e percorre **todas** as máquinas na **mesma
ordem fixa**. No **permutation flow shop** (o caso usual), a **sequência de jobs é a mesma em
todas as máquinas** — basta decidir **uma permutação** dos jobs.

Objetivo: minimizar o **makespan** $C_{\max}$ (término do último job na última máquina).

> **Lugar na série:**
> *máquina única* (uma disjunção) → *job shop* (disjunção por máquina + precedência, ordem de
> máquinas livre por job) → *flow shop* (**mesma ordem de máquinas para todos**). Essa estrutura
> extra permite um MILO **sem big-M** (formulação posicional), em contraste com o job shop.

## Dataset

Arquivos `problem_{m}m_{n}j.csv` com $m \in \{3,5,8,10\}$ máquinas e $n \in \{10,20,30,50,100\}$
jobs (tempos inteiros entre 1 e 20). Cada CSV tem os **jobs nas linhas** e as **máquinas nas
colunas**.

## Preâmbulo: pacotes e solver

In [ ]:
import Pkg

function ensure(pkgs)
    for p in pkgs
        try
            @eval import $(Symbol(p))
        catch
            Pkg.add(p)
        end
    end
end

ensure(["JuMP", "HiGHS", "DataFrames", "CSV", "Plots"])

In [ ]:
using JuMP, HiGHS, DataFrames, CSV, Plots
gr()

const SOLVER = HiGHS.Optimizer

## Leitura das instâncias

A função devolve os nomes dos jobs, os nomes das máquinas e a matriz $P$ de tempos
($n \times m$, linha = job, coluna = máquina).

In [ ]:
function load_fssp(path)
    df = CSV.read(path, DataFrame)          # 1ª coluna = nomes dos jobs
    jobs = string.(df[!, 1])
    machines = names(df)[2:end]
    P = Matrix{Int}(df[:, 2:end])           # n × m
    return jobs, string.(machines), P
end

FSSPDIR = "fssp_problems"                    # ajuste para a pasta das instâncias
jobs, machines, P = load_fssp(joinpath(FSSPDIR, "problem_3m_10j.csv"))
println("instância 3m_10j: ", size(P, 1), " jobs × ", size(P, 2), " máquinas")
P

## Avaliando uma permutação: a recursão do makespan

Para uma **permutação fixa** dos jobs, o escalonamento (semi-ativo) é determinado por uma
recursão $O(n\cdot m)$ sobre os tempos de conclusão acumulados de cada máquina.

In [ ]:
# perm: vetor de índices de jobs (1..n) na ordem de processamento
function makespan(P::Matrix{Int}, perm)
    m = size(P, 2)
    C = zeros(Int, m)
    for j in perm
        C[1] += P[j, 1]
        for i in 2:m
            C[i] = max(C[i], C[i-1]) + P[j, i]
        end
    end
    return C[m]
end

# tabela de início/fim por (job, máquina) — usada no Gantt
function schedule_intervals(P::Matrix{Int}, perm)
    m = size(P, 2)
    Cm = zeros(Int, m)
    iv = Dict{Tuple{Int,Int},Tuple{Int,Int}}()
    for j in perm
        sprev = 0
        for i in 1:m
            start = max(Cm[i], sprev)
            fin = start + P[j, i]
            iv[(j, i)] = (start, fin)
            Cm[i] = fin; sprev = fin
        end
    end
    return iv
end

## Heurísticas

### NEH (Nawaz–Enscore–Ham)

Heurística construtiva clássica e muito eficaz: ordena os jobs por tempo total decrescente e os
insere, um a um, na posição que minimiza o makespan parcial.

In [ ]:
function neh(P::Matrix{Int})
    n, _ = size(P)
    totals = vec(sum(P, dims = 2))
    order = sortperm(totals; rev = true)        # jobs por tempo total decrescente
    seq = [order[1]]
    for idx in 2:n
        j = order[idx]
        best = typemax(Int); bestpos = 1
        for pos in 1:length(seq)+1
            cand = vcat(seq[1:pos-1], j, seq[pos:end])
            cm = makespan(P, cand)
            if cm < best
                best = cm; bestpos = pos
            end
        end
        insert!(seq, bestpos, j)
    end
    return seq, makespan(P, seq)
end

neh_seq, neh_cm = neh(P)
println("NEH: makespan = ", neh_cm)

### Regra de Johnson (exata para $m = 2$)

Para **duas máquinas**, a regra de Johnson dá a sequência **ótima** em tempo $O(n\log n)$:
jobs com $p_{j,1} \le p_{j,2}$ vão para a frente (ordem crescente de $p_{j,1}$); os demais vão
para o fim (ordem decrescente de $p_{j,2}$). Aplicada aqui às duas primeiras máquinas, é apenas
ilustrativa (para $m \ge 3$ o problema é NP-difícil).

In [ ]:
function johnson(P::Matrix{Int})   # usa as 2 primeiras máquinas (exato só p/ m=2)
    n = size(P, 1)
    left = Tuple{Int,Int}[]; right = Tuple{Int,Int}[]
    for j in 1:n
        a, b = P[j, 1], P[j, 2]
        a <= b ? push!(left, (a, j)) : push!(right, (b, j))
    end
    sort!(left)                # crescente por p_{j,1}
    sort!(right; rev = true)   # decrescente por p_{j,2}
    return vcat([j for (_, j) in left], [j for (_, j) in right])
end

johnson(P)

## Modelo exato MILO — formulação posicional (sem big-M)

A estrutura do flow shop permite uma formulação **posicional** (Wilson) limpa, sem big-M.

**Variáveis.** $x_{j,k} \in \{0,1\}$: job $j$ ocupa a posição $k$ da sequência;
$C_{k,i} \ge 0$: conclusão do job da posição $k$ na máquina $i$.

**Atribuição.** cada job em uma posição e cada posição com um job:
$\sum_k x_{j,k} = 1,\quad \sum_j x_{j,k} = 1$.

Seja $\bar p_{k,i} = \sum_j p_{j,i}\,x_{j,k}$ o tempo da posição $k$ na máquina $i$.

**Recursões de conclusão** (sem big-M):
$$
\begin{aligned}
C_{k,i} &\ge C_{k-1,i} + \bar p_{k,i} && \text{(máquina } i \text{ livre só após a posição } k{-}1)\\
C_{k,i} &\ge C_{k,i-1} + \bar p_{k,i} && \text{(job da posição } k \text{ só após a máquina } i{-}1)
\end{aligned}
$$

**Objetivo.** $\min C_{n,m}$.

In [ ]:
function solve_positional(P::Matrix{Int}; time_limit = 60.0)
    n, m = size(P)
    model = Model(SOLVER); set_silent(model)
    set_optimizer_attribute(model, "time_limit", float(time_limit))

    @variable(model, x[1:n, 1:n], Bin)          # x[j,k] = job j na posição k
    @variable(model, C[1:n, 1:m] >= 0)

    @constraint(model, [j = 1:n], sum(x[j, k] for k in 1:n) == 1)
    @constraint(model, [k = 1:n], sum(x[j, k] for j in 1:n) == 1)

    pp(k, i) = sum(P[j, i] * x[j, k] for j in 1:n)   # tempo da posição k na máquina i

    @constraint(model, C[1, 1] >= pp(1, 1))
    for k in 1:n, i in 1:m
        k > 1 && @constraint(model, C[k, i] >= C[k-1, i] + pp(k, i))
        i > 1 && @constraint(model, C[k, i] >= C[k, i-1] + pp(k, i))
    end

    @objective(model, Min, C[n, m])
    optimize!(model)

    seq = zeros(Int, n)
    for k in 1:n, j in 1:n
        value(x[j, k]) > 0.5 && (seq[k] = j)
    end
    return seq, objective_value(model), termination_status(model)
end

seq, cmax, st = solve_positional(P; time_limit = 60.0)
println("MILO: makespan = ", round(Int, cmax), "  | status = ", st)
println("sequência ótima: ", [jobs[j] for j in seq])

## Gráfico de Gantt (em escada)

Cada job percorre $M_1 \to M_2 \to \dots$ na mesma sequência — daí o padrão em cascata.

In [ ]:
const PALETTE = [:steelblue, :orange, :seagreen, :firebrick, :mediumpurple,
                 :sienna, :goldenrod, :teal, :hotpink, :slategray]

function gantt_fssp(P::Matrix{Int}, perm, jobs; title = "")
    m = size(P, 2)
    iv = schedule_intervals(P, perm)
    cmax = maximum(f for ((_, _), (_, f)) in iv)
    pos = Dict(j => k for (k, j) in enumerate(perm))

    plt = plot(; legend = false, yflip = true, xlabel = "Tempo", ylabel = "Máquina",
                 yticks = (1:m, ["M$i" for i in 1:m]), size = (950, 70m + 130), title = title)
    for ((j, i), (s, f)) in iv
        col = PALETTE[(pos[j] - 1) % length(PALETTE) + 1]
        plot!(plt, Shape([s, f, f, s], [i - 0.35, i - 0.35, i + 0.35, i + 0.35]);
              fillcolor = col, linecolor = :white, linewidth = 0.8)
        (f - s) >= 4 && annotate!(plt, (s + f) / 2, i, text(jobs[j], :white, :center, 7))
    end
    vline!(plt, [cmax]; linestyle = :dash, color = :black)
    return plt
end

gantt_fssp(P, seq, jobs; title = "Flow shop 3×10 — makespan = $(round(Int, cmax))")

## Heurística × exato e dificuldade computacional

A formulação posicional tem $n^2$ binárias (independente de $m$), mas o acoplamento das recursões
cresce com $m$. Comparamos **NEH** (rápido) ao **MILO** com limite de tempo. Para instâncias
maiores, o exato tende a não fechar o gap — e pode até **perder para o NEH** dentro do tempo.

In [ ]:
using Printf

testset = ["problem_3m_10j.csv", "problem_5m_10j.csv",
           "problem_5m_20j.csv", "problem_10m_20j.csv"]
println(@sprintf("%-14s %3s %4s %6s %7s %8s %s", "inst", "m", "n", "NEH", "MILO", "t(s)", "status"))
for f in testset
    _, _, Pt = load_fssp(joinpath(FSSPDIR, f))
    _, ncm = neh(Pt)
    t = @elapsed (sq, cm, stt) = solve_positional(Pt; time_limit = 30.0)
    name = replace(f, "problem_" => "", ".csv" => "")
    println(@sprintf("%-14s %3d %4d %6d %7d %8.2f  %s",
            name, size(Pt, 2), size(Pt, 1), round(Int, ncm), round(Int, cm), t, string(stt)))
end

## Leitura para a aula

- O **flow shop** é o job shop com **ordem de máquinas idêntica para todos os jobs** — basta
  decidir **uma permutação**. Essa estrutura rende um MILO **posicional sem big-M**, mais
  "limpo" que o do job shop.
- **NEH** é uma heurística construtiva excelente: barata e frequentemente muito perto (ou igual)
  ao ótimo. Para $m = 2$, a **regra de Johnson** dá o ótimo em tempo polinomial.
- Para $m \ge 3$, o problema é **NP-difícil**: o MILO resolve instâncias pequenas, mas em
  instâncias maiores não fecha o gap no tempo — podendo ficar atrás do NEH.

### Próximos passos sugeridos

1. **Melhoria local** após o NEH (ex.: trocas/inserções, ou *iterated greedy*).
2. **Warm-start** do MILO com a sequência do NEH (`set_start_value` nos $x_{j,k}$).
3. Rodar o dataset inteiro e plotar **tempo do exato × (n, m)** e **gap NEH × ótimo**.